In [1]:
import pandas as pd
import numpy as np
import sys
import os


In [2]:
sys.path.append(os.path.abspath(".."))

In [ ]:
from src.data_utils import (load_data)
from src.cleaning_utils import (drop_irrelevant_columns,
                                standardize_placeholders,
                                to_category,
                                handle_missing_values,
                                log_transform_skewed_columns,
                                cap_outliers_iqr,
                                drop_duplicates,
                                convert_to_datetime,
                                count_unique_values
                                )



In [4]:
X_train = load_data("../data/X_train.csv")
y_train = load_data("../data/y_train.csv")

df_raw = X_train.merge(y_train, on="id")

DataFrame successfully loaded.
Shape: (59400, 40)
DataFrame successfully loaded.
Shape: (59400, 2)


In [ ]:
cols_to_drop = [
    "id",                # unique identifier
    "wpt_name",          # too many unique values
    "num_private",       # mostly zeros / unclear meaning
    "recorded_by",        # constant value
    "scheme_name",       # too many unique values
    "extraction_type_group",  # already captured by extraction_type
    "payment_type",     # already captured by payment
    "quantity_group",   # already captured by quantity
    "waterpoint_type_group",  # already captured by waterpoint_type
    "subvillage"       # too many unique values
]

In [6]:
df_clean = drop_irrelevant_columns(df_raw, cols_to_drop)

Dropped columns: ['id', 'wpt_name', 'num_private', 'recorded_by', 'scheme_name', 'extraction_type_group', 'payment_type']


In [ ]:
df_clean = standardize_placeholders(df_clean)

In [ ]:
df.shape


In [ ]:
df.describe()

In [ ]:
print(df.columns)

In [ ]:
# Check missing values
print(df.isnull().sum())

In [ ]:
numeric_cols = ['gps_height', 'longitude', 'latitude', 
                'num_private', 'population', 'construction_year']

df[numeric_cols] = df[numeric_cols].replace(0, np.nan)

# Numeric columns → mean imputation
for col in numeric_cols:
    df[col] = df[col].fillna(df[col].mean())



# Categorical columns → fill with "Unknown"
categorical_cols = df.select_dtypes(include=['object', 'str', ]).columns.tolist()
categorical_cols.remove('status_group')  # keep target untouched

for col in categorical_cols:
    df[col] = df[col].fillna("Unknown")

# Replace '0' strings or numbers with 'Unknown' for categorical columns
for col in categorical_cols:
    df[col] = df[col].replace(['0', 0], 'special_unknown')

# Convert region_code and district_code to categorical variables
df['region_code'] = df['region_code'].astype('category')
df['district_code'] = df['district_code'].astype('category')

In [ ]:
categorical_cols = df.select_dtypes(include=['object', 'str', 'category']).columns.tolist()
print(df[categorical_cols].columns)

In [ ]:
for col in categorical_cols:
    if (df[col] == '0').any() or (df[col] == 0).any():
        print(
            col,
            df[col].value_counts().reindex(['0', 0]).dropna()
        )

In [ ]:
# Check missing values
print(df.isnull().sum())

In [ ]:
df.drop_duplicates(inplace=True)


In [ ]:
# Convert date_recorded to datetime
df['date_recorded'] = pd.to_datetime(df['date_recorded'])
# Extract year, month, day
df['year_recorded'] = df['date_recorded'].dt.year




In [ ]:
df['status_group'].value_counts()

In [ ]:
for col in df[categorical_cols].columns:
    print(col, df[col].nunique())

In [ ]:
# Drop useless + extreme columns
df.drop(['id', 'recorded_by', 'wpt_name', 'subvillage', 'scheme_name', 'ward', 'region_code', 'district_code'], axis=1, inplace=True)

In [ ]:
top_installers = df['installer'].value_counts().nlargest(10).index

df['installer_grouped'] = df['installer'].apply(
    lambda x: x if x in top_installers else 'Other'
)

In [ ]:
top_funders = df['funder'].value_counts().nlargest(10).index

df['funder_grouped'] = df['funder'].apply(
    lambda x: x if x in top_funders else 'Other'
)

In [ ]:
counts = df['lga'].value_counts()
df['lga_grouped'] = df['lga'].apply(lambda x: x if counts[x] > 150 else 'Other')

In [ ]:
ct = pd.crosstab(df['lga_grouped'], df['status_group'])
ct_sorted = ct.sort_values(by='functional', ascending=False)
ct_sorted

In [ ]:
missing_percent = df.isna().mean() * 100
missing_percent.sort_values(ascending=False)

In [ ]:
# Save cleaned DataFrame
df.to_csv("../data/cleaned_data.csv", index=False)